# Demand-Side Management MPC (DSMPC) Simulations HALL AND ERDIN

Copyright &copy; 2024, Alexander Erdin (aerdin@ethz.ch), ETH Zurich

This project is licensed under the MIT License.

## Setup

In [ ]:
# Reload scripts when executed
%load_ext autoreload
%autoreload 2

In [38]:
import warnings
import casadi
import cvxpy as cp
import numpy as np
import pandas as pd
from time import sleep
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as ticker
from matplotlib.colors import LightSource
from matplotlib.patches import Polygon
from matplotlib.collections import PatchCollection
from matplotlib import colormaps
from params import DSMPCParams
from systems import LinearSystem
from controllers import EMPC, CEMPC
from IPython.display import clear_output
from utils import adjust_margins

In [39]:
# Fix random seed and print options
np.random.seed(1)
np.set_printoptions(threshold=10000, linewidth=np.inf)

## Approximate Infinite-Horizon Overtaking-Optimal Trajectory

In [10]:
# Define period length and number of periods
T, N = 96, 6

# Load parameters and create system with a long horizon
zeta_max = 40 * np.ones([T,10]); zeta_max[23::24,:] = 1
params_inf = DSMPCParams(N=N*T, T=T, zeta_max=zeta_max, zeta_min=-zeta_max)
sys_inf = LinearSystem(params_inf.sys)

# Initialize controller
ctrl_inf = EMPC(sys_inf, params_inf.ctrl, solver='cvxpy')

# Solve EMPC problem with free initial state
sol_u, sol_x, error_msg, dual_vars_inf, _ = ctrl_inf.solve(t=0)
if error_msg != None:
    warnings.warn(error_msg)

# Save trajectory
x_inf = sol_x.T
u_inf = sol_u.T

## Simulate System Without Terminal Ingredients

### Solve DSMPC for Varying Initial States

In [15]:
# Load parameters and create system
params = DSMPCParams(T=96, zeta_max=zeta_max, zeta_min=-zeta_max)
sys = LinearSystem(params.sys)

# Initialize controller
ctrl = EMPC(sys, params.ctrl)

# Setup simulation
num_steps = 48
x_0 = [np.tile(np.array([10/4*i, 15/4*i]), 10) for i in range(5)]
num_traj  = len(x_0)

# Allocate state and input trajectories
x = np.full((num_traj, num_steps+1, sys.n), np.nan)
u = np.full((num_traj, num_steps,   sys.m), np.nan)
x_ol = np.full((num_traj, params.ctrl.N+1, sys.n), np.nan)
u_ol = np.full((num_traj, params.ctrl.N,   sys.m), np.nan)

# Simulate closed-loop system
for i, xi_0 in enumerate(tqdm(x_0, desc='Trajectories', leave=False)):
    # Set initial state
    x[i,0,:] = xi_0

    # Simulate closed-loop system
    for t in tqdm(range(num_steps),desc='     MPC', leave=False):
        # Solve the EMPC problem
        sol_u, sol_x, error_msg, _, _ = ctrl.solve(t=t, x_0=x[i,t,:])
        if error_msg != None:
            warnings.warn(error_msg)
            break
        
        # Save open-loop trajectory
        if t == 0:
            x_ol[i,:] = sol_x.T
            u_ol[i,:] = sol_u.T

        # Propagate dynamics and save input
        u[i,t,:]   = sol_u[:,0]
        x[i,t+1,:] = sys.f(sol_x[:,0], sol_u[:,0], t=t).reshape((sys.n,))
        sleep(0.01)

# Clear cell output
clear_output()

### Solve DSMPC for Varying Horizons

In [16]:
# Setup simulation
num_steps = 48
horizon = [6, 9, 12, 18, 24, 36, 48]
num_horizon = len(horizon)

# Allocate state and input trajectories
x2 = np.full((num_horizon, num_steps+1, sys.n), np.nan)
u2 = np.full((num_horizon, num_steps,   sys.m), np.nan)
x2_ol = np.full((num_horizon, num_steps, max(horizon)+1, sys.n), np.nan)
u2_ol = np.full((num_horizon, num_steps, max(horizon),   sys.m), np.nan)

# Simulate closed-loop system
for i, N in enumerate(tqdm(horizon, desc='Horizons', leave=False)):
    # Load parameters and create system
    params2 = DSMPCParams(N=N, T=96, zeta_max=zeta_max, zeta_min=-zeta_max)
    sys2 = LinearSystem(params2.sys)

    # Initialize controller
    ctrl2 = EMPC(sys2, params2.ctrl)

    # Set initial state
    x2[i,0,:] = np.tile(np.array([10, 15]), 10)

    # Simulate closed-loop system
    for t in tqdm(range(num_steps),desc='   MPC', leave=False):
        # Solve the EMPC problem
        sol_u, sol_x, error_msg, _, _ = ctrl2.solve(t=t, x_0=x2[i,t,:])
        if error_msg != None:
            warnings.warn(error_msg)
            break
        
        # Save open-loop trajectory
        x2_ol[i,t,:N+1,:] = sol_x.T
        u2_ol[i,t,:N,:]   = sol_u.T

        # Propagate dynamics and save input
        u2[i,t,:]   = sol_u[:,0]
        x2[i,t+1,:] = sys2.f(sol_x[:,0], sol_u[:,0], t=t).reshape((sys.n,))
        sleep(0.01)

# Clear cell output
clear_output()

## Simulate System With Terminal Ingredients

### Compute Periodic Reference Trajectory

In [ ]:
# Define period length
T = 96

# Define non-convex cost
gamma_1 = 5 * (1 - 2*np.random.rand(T, 10))

# Load parameters and create periodic system
zeta_max = 40 * np.ones([T,10]); zeta_max[23::24,:] = 1
params_p = DSMPCParams(N=T, T=T, zeta_max=zeta_max, zeta_min=-zeta_max, verbose=False) # gamma_1=gamma_1,
sys_p = LinearSystem(params_p.sys)

# Initialize controller
ctrl_p = CEMPC(sys_p, params_p.ctrl)

# Solve EMPC problem with free initial state
sol_u, sol_x, error_msg, dual_vars_p, stats = ctrl_p.solve(options={'eps_rel': 1E-10, 'eps_abs': 1E-10})
if error_msg != None:
    warnings.warn(error_msg)

# Save trajectory
x_p = sol_x.T
u_p = sol_u.T
dual_vector_p = np.hstack(dual_vars_p[0:ctrl_p.params.N]).T

### Solve DSMPC for Varying Initial States

In [29]:
# Initialize controller
paramst = ctrl_p.params
paramst.N = 24
ctrl = EMPC(sys_p, paramst)

# Setup simulation
num_steps = 48
xt_0 = [np.tile(np.array([10/4*i, 15/4*i]), 10) for i in range(5)]
num_traj  = len(x_0)

# Allocate state and input trajectories
xt = np.full((num_traj, num_steps+1, sys.n), np.nan)
ut = np.full((num_traj, num_steps,   sys.m), np.nan)
xt_ol = np.full((num_traj, paramst.N+1, sys.n), np.nan)
ut_ol = np.full((num_traj, paramst.N,   sys.m), np.nan)

# Simulate closed-loop system
for i, xi_0 in enumerate(tqdm(xt_0, desc='Trajectories', leave=False)):
    # Set initial state
    xt[i,0,:] = xi_0

    # Simulate closed-loop system
    for t in tqdm(range(num_steps),desc='     MPC', leave=False):
        # Solve the EMPC problem
        sol_u, sol_x, error_msg, _, _ = ctrl.solve(t=t, x_0=xt[i,t,:], x_T=x_p[t+paramst.N,:])
        if error_msg != None:
            warnings.warn(error_msg)
            break
        
        # Save open-loop trajectory
        if t == 0:
            xt_ol[i,:] = sol_x.T
            ut_ol[i,:] = sol_u.T

        # Propagate dynamics and save input
        ut[i,t,:]   = sol_u[:,0]
        xt[i,t+1,:] = sys.f(sol_x[:,0], sol_u[:,0], t=t).reshape((sys.n,))
        sleep(0.01)

# Clear cell output
clear_output()

### Solve DSMPC for Varying Horizons

In [30]:
# Setup simulation
num_steps = 48
horizont = [24, 36, 48]
num_horizont = len(horizont)

# Allocate state and input trajectories
xt2 = np.full((num_horizont, num_steps+1, sys.n), np.nan)
ut2 = np.full((num_horizont, num_steps,   sys.m), np.nan)
xt2_ol = np.full((num_horizont, num_steps, max(horizont)+1, sys.n), np.nan)
ut2_ol = np.full((num_horizont, num_steps, max(horizont),   sys.m), np.nan)

# Simulate closed-loop system
for i, N in enumerate(tqdm(horizont, desc='Horizons', leave=False)):
    # Load parameters and create system
    paramst.N = N

    # Initialize controller
    ctrl2 = EMPC(ctrl_p.sys, paramst)

    # Set initial state
    xt2[i,0,:] = np.tile(np.array([10, 15]), 10)

    # Simulate closed-loop system
    for t in tqdm(range(num_steps),desc='   MPC', leave=False):
        # Solve the EMPC problem
        sol_u, sol_x, error_msg, _, _ = ctrl2.solve(t=t, x_0=xt2[i,t,:], x_T=x_p[t+N,:])
        if error_msg != None:
            warnings.warn(error_msg)
            break
        
        # Save open-loop trajectory
        xt2_ol[i,t,:N+1,:] = sol_x.T
        ut2_ol[i,t,:N,:]   = sol_u.T

        # Propagate dynamics and save input
        ut2[i,t,:]   = sol_u[:,0]
        xt2[i,t+1,:] = sys2.f(sol_x[:,0], sol_u[:,0], t=t).reshape((sys.n,))
        sleep(0.01)

# Clear cell output
clear_output()

### Compare Cost

In [ ]:
# Compute cost with and without terminal constraints
data = []
for i in range(num_traj):
    cost, costt = 0, 0
    for k in range(num_steps):
        cost  += params.ctrl.stage_cost(x[i, k],  u[i, k],  t=k)[0]
        costt += params.ctrl.stage_cost(xt[i, k], ut[i, k], t=k)[0]
    
    # Append the computed costs
    data.append([cost, costt, np.abs(cost/costt-1)])

# Create a pandas DataFrame
columns = pd.MultiIndex.from_tuples([('Cost', 'Without'), ('Cost', 'With'), ('Rel. Diff','[%]')])
cost_df = pd.DataFrame(data, columns=columns, index=pd.Index(range(num_traj), name="Trajectory"))
cost_df